# 📦 LDA & QDA
**ISLP Chapter 4 · Pattern Recognition for the Rest of Us**

> Linear and Quadratic Discriminant Analysis use probability theory to classify. Instead of fitting a boundary directly (like logistic regression), they model the distribution of X within each class and use Bayes' theorem to classify.

### What you'll learn
- Bayes' theorem as a classification framework
- LDA — linear decision boundaries (assumes shared covariance)
- QDA — quadratic boundaries (each class has its own covariance)
- When LDA/QDA outperform logistic regression
- Comparing all three: LR vs LDA vs QDA

### Dataset: Stock market direction prediction (Smarket)

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler

In [ ]:
# ── Load datasets from the course repo ──────────────────────────────────────
import subprocess, os

# Clone the course repo if not already present (works in Colab)
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    # Fallback: already in repo root (e.g. running locally)
    DATA_DIR = '../data'

print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Available datasets: {os.listdir(DATA_DIR)}")

### Load Dataset: Smarket

In [ ]:
import pandas as pd
smarket = pd.read_csv(f'{DATA_DIR}/Smarket.csv')
smarket['Direction_num'] = (smarket['Direction'] == 'Up').astype(int)
print(f"Smarket: {smarket.shape}  |  Up days: {smarket['Direction_num'].mean():.1%}")
smarket.head()

## 📐 Part 1 — Bayes' Theorem for Classification

LDA and QDA both use Bayes' theorem:

```
P(Y=k | X=x) = P(X=x | Y=k) × P(Y=k) / P(X=x)
             =   likelihood  ×   prior   / evidence
```

The class with the highest posterior probability wins.

**LDA** assumes Xₖ ~ N(μₖ, Σ) — each class has its own mean but **shared covariance** Σ. This gives *linear* decision boundaries.

**QDA** assumes Xₖ ~ N(μₖ, Σₖ) — each class has its own covariance Σₖ. This gives *quadratic* boundaries. More flexible but needs more data to estimate.

In [ ]:
# Visualize LDA vs QDA decision boundaries
from sklearn.datasets import make_classification
np.random.seed(42)

# Create two scenarios
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scenario 1: Equal covariance → LDA is optimal
X1 = np.vstack([np.random.multivariate_normal([0,0], [[1,0.5],[0.5,1]], 150),
                np.random.multivariate_normal([2,2], [[1,0.5],[0.5,1]], 150)])
y1 = np.array([0]*150+[1]*150)

# Scenario 2: Unequal covariance → QDA wins
X2 = np.vstack([np.random.multivariate_normal([0,0], [[0.5,0],[0,2]], 150),
                np.random.multivariate_normal([2,2], [[2,0],[0,0.5]], 150)])
y2 = np.array([0]*150+[1]*150)

for ax, X, y, title in zip(axes, [X1,X2], [y1,y2],
                            ['Equal covariances → LDA wins', 'Unequal covariances → QDA wins']):
    lda = LinearDiscriminantAnalysis().fit(X, y)
    qda = QuadraticDiscriminantAnalysis().fit(X, y)
    
    xx, yy = np.meshgrid(np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 200),
                          np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    
    Z_lda = lda.predict(grid).reshape(xx.shape)
    ax.contour(xx, yy, Z_lda, colors=['#1e5fa8'], linewidths=2, linestyles='-')
    
    Z_qda = qda.predict(grid).reshape(xx.shape)
    ax.contour(xx, yy, Z_qda, colors=['#e85d20'], linewidths=2, linestyles='--')
    
    ax.scatter(X[y==0,0], X[y==0,1], color='#1e5fa8', alpha=0.4, s=15)
    ax.scatter(X[y==1,0], X[y==1,1], color='#e85d20', alpha=0.4, s=15)
    
    from matplotlib.lines import Line2D
    ax.legend([Line2D([0],[0],color='#1e5fa8',lw=2),
               Line2D([0],[0],color='#e85d20',lw=2,ls='--')],
              [f'LDA (acc={lda.score(X,y):.2f})',
               f'QDA (acc={qda.score(X,y):.2f})'], fontsize=9)
    ax.set_title(title)

plt.suptitle('LDA vs QDA Decision Boundaries', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 LDA boundary is always a straight line. QDA can curve to fit unequal covariances.")

In [ ]:
# Compare LR, LDA, QDA on Smarket
predictors = ['Lag1','Lag2','Lag3','Lag4','Lag5','Volume']
X = smarket[predictors].values
y = smarket['Direction_num'].values

# Train on 2001-2004, test on 2005 (temporal split — important for market data!)
mask_train = smarket['Year'] < 2005
X_tr, X_te = X[mask_train], X[~mask_train]
y_tr, y_te = y[mask_train], y[~mask_train]

print(f"Train: {X_tr.shape[0]} days (2001-2004)")
print(f"Test: {X_te.shape[0]} days (2005)")

models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'LDA':                 LinearDiscriminantAnalysis(),
    'QDA':                 QuadraticDiscriminantAnalysis(),
}

print("\n=== Accuracy on 2005 Test Data ===")
for name, model in models.items():
    model.fit(X_tr, y_tr)
    acc = model.score(X_te, y_te)
    print(f"  {name:<25} {acc:.4f}")

print("\n📌 All three methods are comparable here — the stock market is hard to predict!")
print("   Baseline (always predict 'Up'): {:.4f}".format(y_te.mean()))

In [ ]:
# LDA's discriminant scores — visualize the separation
lda = LinearDiscriminantAnalysis()
lda.fit(X_tr, y_tr)

scores_tr = lda.transform(X_tr)
scores_te = lda.transform(X_te)

fig, ax = plt.subplots(figsize=(9, 4))
for split, scores, y_split, label, alpha in [
    (None, scores_tr, y_tr, 'Train', 0.4),
    (None, scores_te, y_te, 'Test', 0.9)]:
    ax.hist(scores[y_split==0, 0], bins=25, alpha=alpha*0.7,
            color='#1e5fa8', density=True,
            label=f'Down ({label})' if alpha > 0.5 else None)
    ax.hist(scores[y_split==1, 0], bins=25, alpha=alpha*0.7,
            color='#e85d20', density=True,
            label=f'Up ({label})' if alpha > 0.5 else None)

ax.axvline(0, color='black', lw=1.5, ls='--', label='Decision boundary')
ax.set_xlabel('LDA Discriminant Score')
ax.set_ylabel('Density')
ax.set_title('LDA Discriminant Scores — Up vs Down Days\n(overlap = hard classification problem)')
ax.legend()
plt.tight_layout()
plt.show()
print("\n📌 Large overlap between Up/Down scores confirms stock market is hard to predict from lagged returns")

In [ ]:
# @title 📝 Quiz — LDA & QDA
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What distributional assumption does LDA make about the features?
# @markdown **Q2:** What is the key difference between LDA and QDA?
# @markdown **Q3:** When does LDA outperform logistic regression?
# @markdown **Q4:** Why do we need a prior P(Y=k) in Bayes' theorem?
# @markdown **Q5:** If your data has 5 classes and 2 predictors, how many discriminant functions does LDA produce?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "LDA & QDA"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username, then click ▶ Run for question-by-question feedback.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re, time
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_quiz_questions():
    """Pull question text from the quiz cell @markdown lines."""
    try:
        import ipynbname
        nb_path = ipynbname.path()
        with open(nb_path) as f:
            nb = json.load(f)
        for cell in nb["cells"]:
            src = "".join(cell.get("source", []))
            if "@title" in src and "Quiz" in src:
                qs = re.findall(r"@markdown \*\*Q\d+:\*\* (.+)", src)
                if qs: return qs
    except Exception:
        pass
    return []

def _call_gemini(prompt, max_retries=3):
    """Call Gemini with retry on 429 rate limit."""
    last_err = None
    for attempt in range(max_retries):
        try:
            import google.generativeai as genai
            import google.auth, google.auth.transport.requests
            creds, _ = google.auth.default()
            creds.refresh(google.auth.transport.requests.Request())
            genai.configure(credentials=creds)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(
                prompt,
                generation_config={"max_output_tokens": 1500, "temperature": 0.3}
            )
            return result.text, "Gemini via Colab"
        except Exception as e:
            last_err = str(e)
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 2 ** attempt
                print(f"  Rate limit hit — waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
                continue
            break
    # Try stored key as fallback
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini via key"
    except Exception:
        pass
    return None, last_err

def _build_prompt(answers_dict, nb_name, questions):
    answered  = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total   = len(answers_dict)
    n_done    = len(answered)

    # Pair each question with the student answer
    qa_pairs = ""
    for i, (k, v) in enumerate(answers_dict.items(), 1):
        q_text = questions[i-1] if i-1 < len(questions) else f"Question {i}"
        a_text = str(v).strip() if str(v).strip() else "(no answer)"
        qa_pairs += f"Q{i}: {q_text}\nA{i}: {a_text}\n\n"

    return f"""You are a TA grading a student quiz for the "{nb_name}" notebook in a data science course called "Data Pattern Recognition for the Rest of Us" (based on ISLP and business analytics).

{qa_pairs.strip()}

For EACH question:
- Decide if the answer is CORRECT, PARTIALLY CORRECT, or INCORRECT
- A short paraphrase or reasonable approximation counts as CORRECT — do not require exact wording
- For INCORRECT or PARTIAL: name the specific concept they need to review (1 sentence)
- For CORRECT: brief confirmation of what they got right (1 sentence)

Then give an overall summary.

Respond ONLY in this exact JSON format (no markdown fences, no extra text):
{{
  "questions": [
    {{
      "q": 1,
      "status": "<CORRECT|PARTIAL|INCORRECT>",
      "comment": "<one specific sentence>"
    }}
  ],
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent|Good|Needs Review|Incomplete>",
  "summary": "<2 sentences overall: strengths and what to revisit>",
  "review_tip": "<one specific concept, chapter, or notebook to review if they struggled — or 'Great work!' if excellent>"
}}

Scoring guide: CORRECT=1pt, PARTIAL=0.5pt (round to nearest int), INCORRECT=0pt."""

def _show(result, username, nb_name, source, questions):
    STATUS_ICON  = {"CORRECT": "\u2705", "PARTIAL": "\u26a0\ufe0f", "INCORRECT": "\u274c"}
    STATUS_COLOR = {"CORRECT": "\033[92m", "PARTIAL": "\033[93m", "INCORRECT": "\033[91m"}
    R = "\033[0m"
    grade = result.get("grade", "?")
    GRADE_COLOR = {"Excellent":"\033[92m","Good":"\033[94m",
                   "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    GC = GRADE_COLOR.get(grade, "")
    n  = len(answers)
    qs = result.get("quiz_score", 0)
    bar = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*60)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*60)
    print(f"  Student  : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade    : {GC}{grade}{R}")
    print(f"  Score    : {qs}/{n}  [{bar}]")
    print()

    # Question-by-question breakdown
    q_results = result.get("questions", [])
    if q_results:
        print("  \u2500"*29)
        for qr in q_results:
            idx    = qr.get("q", 0) - 1
            status = qr.get("status", "?")
            icon   = STATUS_ICON.get(status, "\u2753")
            color  = STATUS_COLOR.get(status, "")
            comment= qr.get("comment", "")
            q_text = questions[idx] if idx < len(questions) else f"Question {qr.get('q','?')}"
            # Truncate long question text for display
            q_short = q_text[:55] + "..." if len(q_text) > 55 else q_text
            print(f"  {icon} {color}Q{qr.get('q','?')} {status}{R}")
            print(f"     {q_short}")
            if comment:
                for line in textwrap.wrap(comment, 54):
                    print(f"     \u2192 {line}")
            print()

    # Summary
    summary = result.get("summary", "")
    if summary:
        print("  \u2500"*29)
        print("  Overall:")
        for line in textwrap.wrap(summary, 56):
            print(f"  {line}")

    # Review tip
    tip = result.get("review_tip", "")
    if tip and tip != "Great work!":
        print()
        for line in textwrap.wrap(f"\U0001f4d6 Review: {tip}", 56):
            print(f"  {line}")
    elif tip == "Great work!":
        print()
        print("  \U0001f4d6 Great work! Keep going.")

    print("\u2550"*60 + "\n")

def _fallback_grade(answers_dict):
    """Smart fallback — grade on completion only, no length penalty."""
    n  = len(answers_dict)
    nd = sum(1 for v in answers_dict.values() if str(v).strip())
    if nd == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "questions":[],
                "summary":"No answers provided — fill in the quiz above.",
                "review_tip":"Complete the quiz and re-run for AI feedback."}
    elif nd == n:
        return {"quiz_score":n,"grade":"Good",
                "questions":[],
                "summary":f"All {n} questions answered. AI grading was unavailable — re-run to get question-by-question feedback.",
                "review_tip":"Re-run this cell in a few minutes for detailed AI feedback."}
    else:
        return {"quiz_score":nd,"grade":"Needs Review",
                "questions":[],
                "summary":f"{nd}/{n} questions answered. Complete the remaining {n-nd} and re-run.",
                "review_tip":"Answer all questions for full feedback."}

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd       = sum(1 for v in answers.values() if str(v).strip())
    username = GITHUB_USERNAME.strip()
    questions = _get_quiz_questions()

    print(f"\n  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    if username: print(f"  Student  : @{username}")
    print(f"  Grading  : please wait...\n")

    prompt     = _build_prompt(answers, _NB_TITLE, questions)
    raw, src   = _call_gemini(prompt)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            nd2    = sum(1 for v in answers.values() if str(v).strip())
            result = {"quiz_score":nd2,"grade":"Good" if nd2==len(answers) else "Needs Review",
                      "questions":[],"summary":raw[:400],"review_tip":""}
    else:
        if src: print(f"  \u26a0 Gemini unavailable ({src[:60]}) \u2014 showing completion feedback\n")
        result = _fallback_grade(answers)

    _show(result, username, _NB_TITLE, src if raw else None, questions)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")
